# Análise de Infraestrutura Computacional
**Projeto:** Análise da Capacidade Computacional e Distribuição de Processadores  
**Disciplina:** Tópicos de Big Data em Python  
**Empresa:** Origem Energia  
**Base de dados:** Inventario.xlsx — 991 registros  

---
## Etapas
1. Importação das bibliotecas
2. Leitura da base original
3. Tratamento e criação de colunas derivadas
4. Dado simulado - Impacto_Operacional
5. Dado simulado - Custo_Upgrade_BRL
6. Visão geral e KPIs
7. Exportação da base tratada

---
## 1. Importação das bibliotecas

In [ ]:
import pandas as pd
import re

---
## 2. Leitura da base original

A base original contém 7 colunas, incluindo a coluna **Setor**, que já estava presente na planilha fornecida pela Origem Energia e indica o departamento ao qual cada equipamento está alocado.

In [ ]:
df = pd.read_excel('../ingest/Inventario.xlsx')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head()

In [ ]:
# Verificar valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())

In [ ]:
# Distribuição da coluna Setor — dado original da planilha
print('Distribuição por Setor:')
print(df['Setor'].value_counts())

---
## 3. Tratamento dos dados
### 3.1 Converter RAM de MB para GB

In [ ]:
df['RAM_GB'] = df['RAM (MB)'] / 1024

print('Distribuição de RAM (GB):')
print(df['RAM_GB'].value_counts().sort_index())

### 3.2 Extrair família de CPU

In [ ]:
def get_cpu_family(cpu_str):
    """
    Extrai a família do processador a partir da string bruta do campo CPU type.
    Retorna categorias como: Xeon Silver, Core i5, Ryzen 5, etc.
    """
    cpu = cpu_str.upper()
    if 'XEON' in cpu:
        if 'PLATINUM' in cpu: return 'Xeon Platinum'
        if 'GOLD' in cpu:     return 'Xeon Gold'
        if 'SILVER' in cpu:   return 'Xeon Silver'
        return 'Xeon Outro'
    if 'RYZEN 9' in cpu: return 'Ryzen 9'
    if 'RYZEN 7' in cpu: return 'Ryzen 7'
    if 'RYZEN 5' in cpu: return 'Ryzen 5'
    if 'RYZEN 3' in cpu: return 'Ryzen 3'
    if 'I9' in cpu: return 'Core i9'
    if 'I7' in cpu: return 'Core i7'
    if 'I5' in cpu: return 'Core i5'
    if 'I3' in cpu: return 'Core i3'
    return 'Outro'

df['CPU_Family'] = df['CPU type'].apply(get_cpu_family)

print('Distribuição por família de CPU:')
print(df['CPU_Family'].value_counts())

### 3.3 Extrair número de núcleos por CPU e calcular total

In [ ]:
def get_cores_per_cpu(cpu_str):
    """
    Extrai a quantidade de núcleos por CPU a partir do padrão [X core(s)]
    presente na string da coluna CPU type.
    """
    match = re.search(r'\[(\d+)\s+core', cpu_str, re.IGNORECASE)
    return int(match.group(1)) if match else 1

df['Cores_per_CPU'] = df['CPU type'].apply(get_cores_per_cpu)

# Total de núcleos = núcleos por CPU × quantidade de CPUs físicas
df['Total_Cores'] = df['Cores_per_CPU'] * df['CPU number']

print('Estatísticas de núcleos totais:')
print(df['Total_Cores'].describe())

### 3.4 Classificar perfil da máquina (Machine_Profile)

In [ ]:
MOBILE_SUFFIXES = [
    '1255U', '1235U', '1035G1', '1135G7',
    '5500U', '5875U', '7735HS', '12700H', '11800H'
]

def get_machine_profile(row):
    """
    Classifica a máquina como Servidor, Notebook ou Desktop
    com base no sistema operacional e no modelo de CPU.
    Nota: classificação estimada — a base não possui campo explícito de tipo de equipamento.
    """
    os_str = str(row['Operating system']).upper()
    cpu_str = str(row['CPU type']).upper()

    if 'SERVER' in os_str or 'UBUNTU' in os_str:
        return 'Servidor'
    if any(suffix in cpu_str for suffix in MOBILE_SUFFIXES):
        return 'Notebook'
    if 'WINDOWS 11' in os_str or 'WINDOWS 10' in os_str:
        return 'Desktop'
    return 'Indefinido'

df['Machine_Profile'] = df.apply(get_machine_profile, axis=1)

print('Distribuição por perfil da máquina:')
print(df['Machine_Profile'].value_counts())

---
## 4. Impacto_Operacional

Indicador estimado de impacto operacional criado com base nas especificações de hardware.  
Cada máquina recebe uma pontuação com base em RAM, núcleos e família de CPU.

| Critério | Pontos |
|---|---|
| RAM ≥ 32 GB | +3 |
| RAM entre 16 e 31 GB | +2 |
| RAM < 16 GB | +1 |
| Total de núcleos ≥ 8 | +3 |
| Total de núcleos entre 4 e 7 | +2 |
| Total de núcleos < 4 | +1 |
| CPU Xeon Platinum / Gold | +3 |
| CPU Xeon Silver / Core i7 / Core i9 / Ryzen 7 / Ryzen 9 | +2 |
| Demais CPUs | +1 |

**Classificação:** Score ≥ 7 → Baixo | Score 5-6 → Médio | Score ≤ 4 → Alto

In [ ]:
ALTA_PERFORMANCE = ['Xeon Platinum', 'Xeon Gold']
MEDIA_PERFORMANCE = ['Xeon Silver', 'Core i7', 'Core i9', 'Ryzen 7', 'Ryzen 9']

def get_impacto_operacional(row):
    score = 0
    if row['RAM_GB'] >= 32:   score += 3
    elif row['RAM_GB'] >= 16: score += 2
    else:                     score += 1
    if row['Total_Cores'] >= 8:   score += 3
    elif row['Total_Cores'] >= 4: score += 2
    else:                         score += 1
    if row['CPU_Family'] in ALTA_PERFORMANCE:    score += 3
    elif row['CPU_Family'] in MEDIA_PERFORMANCE: score += 2
    else:                                        score += 1
    if score >= 7:   return 'Baixo'
    elif score >= 5: return 'Médio'
    else:            return 'Alto'

df['Impacto_Operacional'] = df.apply(get_impacto_operacional, axis=1)

print('Distribuição por Impacto Operacional:')
print(df['Impacto_Operacional'].value_counts())

---
## 5. Dado simulado - Custo_Upgrade_BRL

Estimativa do custo necessário para atualizar cada máquina classificada com **Impacto_Operacional = Alto** para um padrão mínimo aceitável.  
Máquinas com impacto Médio ou Baixo não recebem custo (já estão dentro do padrão).

**Critério de upgrade mínimo por família de CPU:**

| CPU atual | Upgrade para | Custo estimado (R$) |
|---|---|---|
| Core i3 | Core i5 | R$ 1.200 |
| Core i5 | Core i7 | R$ 1.800 |
| Ryzen 3 | Ryzen 5 | R$ 1.100 |
| Ryzen 5 | Ryzen 7 | R$ 1.600 |
| Xeon Outro | Xeon Silver | R$ 4.500 |
| Core i7 / Ryzen 7 com Alto | Upgrade de RAM | R$ 800 |
| Outro | Upgrade básico | R$ 900 |
| Demais (Médio ou Baixo) | Sem upgrade | R$ 0 |

In [ ]:
CUSTO_UPGRADE = {
    'Core i3':     1200,
    'Core i5':     1800,
    'Ryzen 3':     1100,
    'Ryzen 5':     1600,
    'Xeon Outro':  4500,
    'Core i7':      800,
    'Ryzen 7':      800,
    'Core i9':        0,
    'Xeon Silver':    0,
    'Xeon Gold':      0,
    'Xeon Platinum':  0,
    'Outro':        900,
}

def get_custo_upgrade(row):
    """
    Retorna o custo estimado de upgrade da máquina em BRL.
    Apenas máquinas com Impacto_Operacional = Alto recebem custo.
    """
    if row['Impacto_Operacional'] != 'Alto':
        return 0
    return CUSTO_UPGRADE.get(row['CPU_Family'], 0)

df['Custo_Upgrade_BRL'] = df.apply(get_custo_upgrade, axis=1)

print('Distribuição de custos:')
print(df['Custo_Upgrade_BRL'].value_counts().sort_index())
print(f'\nMáquinas que precisam de upgrade: {(df["Custo_Upgrade_BRL"] > 0).sum()}')
print(f'Custo total estimado de upgrade: R$ {df["Custo_Upgrade_BRL"].sum():,.2f}')
print(f'\nCusto de upgrade por setor:')
print(df.groupby('Setor')['Custo_Upgrade_BRL'].sum().sort_values(ascending=False).apply(lambda x: f'R$ {x:,.2f}'))

---
## 6. Visão geral e KPIs

In [ ]:
print(f'Total de registros: {len(df)}')
print(f'Colunas finais: {list(df.columns)}')
df.head(10)

In [ ]:
# KPI 1 — CPU mais utilizada
kpi_cpu = df['CPU_Family'].value_counts().idxmax()
kpi_cpu_pct = df['CPU_Family'].value_counts(normalize=True).max() * 100
print(f'KPI 1 — CPU mais utilizada: {kpi_cpu} ({kpi_cpu_pct:.1f}% da infraestrutura)')

# KPI 2 — % de máquinas com risco operacional alto
kpi_alto = (df['Impacto_Operacional'] == 'Alto').mean() * 100
print(f'KPI 2 — Máquinas com risco operacional alto: {kpi_alto:.1f}%')

# Dado simulado — custo total
print(f'\nCusto total estimado de upgrade: R$ {df["Custo_Upgrade_BRL"].sum():,.2f}')

# Análise extra — setor x impacto
print('\nMáquinas de alto impacto por setor:')
print(df[df['Impacto_Operacional'] == 'Alto']['Setor'].value_counts())

---
## 7. Exportar base tratada

In [ ]:
df.to_csv('../store/processed/inventario_tratado.csv', index=False, encoding='utf-8-sig')
print('CSV exportado: store/processed/inventario_tratado.csv')
print(f'Shape final: {df.shape}')
print(f'Colunas: {list(df.columns)}')